# S6E9 — refreshed ensemble (24 September 2026)

## Result and purpose

A published historical prediction has a reported public ROC-AUC of **0.94657**, above our earlier result and the requested 0.94644. This notebook verifies and exports the exact scored file. If our earlier Kaggle notebook is attached as a third input, it also produces two experimental blends. Those blends have **no measured Kaggle score** until submitted.

### Kaggle inputs to attach

1. Competition data: `playground-series-s6e9`.
2. Notebook output: [S6E9: Does Breaking Ties Help?](https://www.kaggle.com/code/chinzorigtganbat/s6e9-does-breaking-ties-help/output), version **8**. Attach it through **Add Input → Notebooks**.
3. Optional, for the two new blends: output of our earlier **S6E9 | Kaggle-ready 0.94644 Micro-Blend** notebook, containing `submission_ref43_transformer055.csv`.

Run all cells. The exact reference submission is always created. The optional blends are created when the third input is present.

## Context and methods

[Chinzo's version 8 notebook](https://www.kaggle.com/code/chinzorigtganbat/s6e9-does-breaking-ties-help) exports the exact prediction file from [F.A.Nina's historical version 4 blend](https://www.kaggle.com/code/nina2025/ps-s6e9-a-simple-50-50-two-level-straight-blend?scriptVersionId=351902031), reported at 0.94657. We use the exported predictions with attribution. That score belongs to the public source, not a newly trained model here.

Our earlier notebook blended [jazivxt's scored prediction collection](https://www.kaggle.com/code/jazivxt/single-model-zoom-zoom) with a small contribution from [Lâm Huy's Transformer + GBDT notebook](https://www.kaggle.com/code/lamhuy8904/s6e9-94-6-transformer-and-gbdt-ensemble). If its pinned output is attached, this notebook tests 10% and 15% contributions to the 0.94657 reference.

### Key assumptions

- SHA-256 checks identify the exact historical files and prevent silent version changes.
- Public ROC-AUC comes from only part of the test set; small gains can reverse on the private leaderboard.
- Rank correlation on unlabeled test rows describes diversity, not model quality. The new blends are experiments.

Thanks to **F.A.Nina, Chinzo, jazivxt, and Lâm Huy** for sharing their work.

In [ ]:
from io import BytesIO
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
from scipy.stats import rankdata

TARGET = 'Will_Buy_EV'
ID = 'id'
INPUT_ROOT = Path('/kaggle/input')
OUTPUT_ROOT = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()

def matches_for(filename, parent_hint=None):
    roots = [INPUT_ROOT] if INPUT_ROOT.is_dir() else [Path.cwd()]
    matches = []
    for root in roots:
        matches.extend(root.rglob(filename))
    if parent_hint:
        matches = [path for path in matches if parent_hint in str(path).lower()]
    return sorted(set(path.resolve() for path in matches if path.is_file()))

def find_one(filename, parent_hint=None):
    matches = matches_for(filename, parent_hint)
    if len(matches) != 1:
        raise FileNotFoundError(
            f'Expected exactly one {filename} with hint {parent_hint!r}; found {matches[:8]}. '
            'Attach the competition and Chinzo notebook version 8.'
        )
    return matches[0]

sample_path = find_one('sample_submission.csv')
chinzo_path = (
    find_one('submission.csv', 's6e9-does-breaking-ties-help')
    if INPUT_ROOT.is_dir() else find_one('public_chinzo_v8_94657.csv')
)
earlier_matches = matches_for('submission_ref43_transformer055.csv')
earlier_path = earlier_matches[0] if len(earlier_matches) == 1 else None
if len(earlier_matches) > 1:
    raise ValueError(f'Multiple earlier candidate files: {earlier_matches}')

print('Exact 0.94657 source:', chinzo_path)
print('Earlier candidate:', earlier_path or 'not attached; experimental blends will be skipped')

## Validate inputs

Checks cover exact file hashes, competition row count, unique IDs, and prediction alignment.

In [ ]:
EXPECTED_HASHES = {
    'chinzo_v8': 'b625edbaa8a715b3fd69d90fe1f23c9260bdd9cf95b975e04448449b09ab928e',
    'earlier_candidate': '726fac2c1d799d55ebaa2ca3c6366f59fa90c049daeb179b55ce715a188e7894',
}

def verified_csv(name, raw):
    actual = hashlib.sha256(raw).hexdigest()
    if actual != EXPECTED_HASHES[name]:
        raise ValueError(f'{name}: unexpected SHA-256 {actual}')
    frame = pd.read_csv(BytesIO(raw))
    if list(frame.columns) != [ID, TARGET]:
        raise ValueError(f'{name}: unexpected columns {list(frame.columns)}')
    if not frame[ID].is_unique or not np.isfinite(frame[TARGET]).all():
        raise ValueError(f'{name}: invalid IDs or predictions')
    return frame

sample = pd.read_csv(sample_path)
chinzo_raw = chinzo_path.read_bytes()
chinzo = verified_csv('chinzo_v8', chinzo_raw)
earlier = verified_csv('earlier_candidate', earlier_path.read_bytes()) if earlier_path else None

for name, frame in [('chinzo', chinzo), ('earlier', earlier)]:
    if frame is not None and not frame[ID].equals(sample[ID]):
        raise ValueError(f'{name}: IDs do not match the competition sample')

assert len(sample) == 286571
print(f'Validated {len(sample):,} rows and available source hashes.')
print('Unique Chinzo scores:', chinzo[TARGET].nunique())

## Diversity check

ROC-AUC depends on ordering. The exact 0.94657 file has no tied predictions, so tie breaking has no effect. When the optional earlier file is attached, we report its rank correlation with the new source.

In [ ]:
def ranks(values):
    return (rankdata(np.asarray(values), method='average') - .5) / len(values)

r_chinzo = ranks(chinzo[TARGET].to_numpy(float))
r_earlier = ranks(earlier[TARGET].to_numpy(float)) if earlier is not None else None

print('Ties in exact 0.94657 source:', len(r_chinzo) - len(np.unique(r_chinzo)))
if r_earlier is not None:
    print('Rank correlation with our earlier output:',
          round(float(np.corrcoef(r_chinzo, r_earlier)[0, 1]), 9))

## Export submissions

`submission_exact_public_94657.csv` preserves the historical file byte for byte. Submit it first. If the optional earlier output is available, the two additional files are new rank blends with 10% and 15% weight. Their scores are unknown.

In [ ]:
outputs = {
    'submission_exact_public_94657.csv': chinzo[TARGET].to_numpy(float),
}
if r_earlier is not None:
    outputs['submission_chinzo90_earlier10.csv'] = .90 * r_chinzo + .10 * r_earlier
    outputs['submission_chinzo85_earlier15.csv'] = .85 * r_chinzo + .15 * r_earlier

audit_rows = []
for filename, prediction in outputs.items():
    submission = sample[[ID]].copy()
    submission[TARGET] = prediction
    assert submission[ID].equals(sample[ID]) and submission[ID].is_unique
    assert np.isfinite(prediction).all()
    assert (0 <= prediction).all() and (prediction <= 1).all()
    output_path = OUTPUT_ROOT / filename
    if filename == 'submission_exact_public_94657.csv':
        output_path.write_bytes(chinzo_raw)
        assert hashlib.sha256(output_path.read_bytes()).hexdigest() == EXPECTED_HASHES['chinzo_v8']
    else:
        submission.to_csv(output_path, index=False)
    audit_rows.append({
        'file': filename,
        'rows': len(submission),
        'unique_scores': len(np.unique(prediction)),
        'sha256': hashlib.sha256(output_path.read_bytes()).hexdigest(),
    })

audit = pd.DataFrame(audit_rows)
display(audit)
print('Recommended first submission:', OUTPUT_ROOT / 'submission_exact_public_94657.csv')

## Takeaways

The exact source export has a reported public score of 0.94657 and exceeds the requested 0.94644 threshold. Your own submission should confirm it on Kaggle. The optional blends test whether our earlier ranking adds value; they have no measured score yet.

In [ ]:
report = {
    'as_of': '2026-09-24',
    'target_public_auc': .94644,
    'source_reported_public_auc': .94657,
    'source_notebook': 'https://www.kaggle.com/code/chinzorigtganbat/s6e9-does-breaking-ties-help',
    'original_blend_notebook': 'https://www.kaggle.com/code/nina2025/ps-s6e9-a-simple-50-50-two-level-straight-blend?scriptVersionId=351902031',
    'input_hashes': {key: value for key, value in EXPECTED_HASHES.items()
                     if key == 'chinzo_v8' or earlier is not None},
    'outputs': audit.to_dict(orient='records'),
    'measured_new_blend_scores': None,
}
report_path = OUTPUT_ROOT / 'ev_purchase_20260924_report.json'
report_path.write_text(json.dumps(report, indent=2), encoding='utf-8')
print('Wrote', report_path)